In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, os, shutil, subprocess, sys, time, traceback
import numpy as np
import pandas as pd

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
DATA = WORK / 'day11-cbs-data'
PROJECT = WORK / 'day11-cbs-project-src'

# Find the largest attached checkpoint containing aligned product/features/sequences.
data_candidates = []
for fp in INPUT.rglob('features.csv'):
    root = fp.parent
    if not all((root / name).is_file() for name in ('manifest.csv', 'series.npy')):
        continue
    try:
        ids = pd.read_csv(fp, usecols=['source_id'], dtype={'source_id': 'string'})['source_id'].astype('string')
        valid = ids.str.fullmatch(r'\d{8,16}', na=False)
        data_candidates.append((len(ids), int(ids[valid].nunique()), root))
    except Exception as exc:
        print('Skipping unreadable data candidate:', root, type(exc).__name__, str(exc)[:160])
if not data_candidates:
    raise FileNotFoundError('Attach the Day 10 or 20K output containing manifest.csv, features.csv, and series.npy.')
rows, n_tics, DATA_SOURCE = max(data_candidates, key=lambda item: (item[0], item[1], str(item[2])))
if rows < 19_000:
    raise RuntimeError(f'Found only {rows:,} feature rows. Attach the successful 20K Day 10 output.')

posterior_hits = list(INPUT.rglob('row_4923_posterior_by_chain.npz'))
if not posterior_hits:
    raise FileNotFoundError('Attach the successful Day 9.19 output containing row_4923_posterior_by_chain.npz.')
POSTERIOR_PATH = max(posterior_hits, key=lambda p: (p.stat().st_size, str(p)))
project_hits = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/cosmic_black_swan/cli.py').is_file()]
if not project_hits:
    raise FileNotFoundError('Attach Cosmic Swan 4 project source containing pyproject.toml and src/cosmic_black_swan/.')
PROJECT_SOURCE = max(project_hits, key=lambda p: len(list((p / 'src/cosmic_black_swan').glob('*.py'))))

DATA.mkdir(parents=True, exist_ok=True)
for name in ('manifest.csv', 'features.csv', 'series.npy', 'labels.csv', 'feature_errors.csv'):
    src = DATA_SOURCE / name
    if src.exists(): shutil.copy2(src, DATA / name)
for name in ('scores', 'models', 'catalogs', 'pixel_audits', 'dossiers'):
    src = DATA_SOURCE / name
    if src.is_dir():
        shutil.copytree(src, DATA / name, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
shutil.copytree(PROJECT_SOURCE, PROJECT, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '.git'))
sys.path.insert(0, str(PROJECT / 'src'))

features = pd.read_csv(DATA / 'features.csv', dtype={'source_id': 'string'})
series = np.load(DATA / 'series.npy', mmap_mode='r')
if len(features) != len(series): raise ValueError(f'features/series misalignment: {len(features)} vs {len(series)}')
print('Selected data:', DATA_SOURCE)
print('Project source:', PROJECT_SOURCE)
print('Day 9.19 posterior:', POSTERIOR_PATH)
print(f'Checkpoint: {len(features):,} observations / {features.source_id.nunique():,} source IDs; sequence shape={series.shape}')

# Read the Kaggle secret label used by this notebook, with the canonical name as fallback.
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
SECRET_LABEL_USED = 'environment' if GEMINI_API_KEY else None
if not GEMINI_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        secret_client = UserSecretsClient()
        for secret_label in ('Gemini-Api-Key', 'GEMINI_API_KEY'):
            try:
                candidate_key = secret_client.get_secret(secret_label) or ''
            except Exception:
                candidate_key = ''
            if candidate_key:
                GEMINI_API_KEY = candidate_key
                SECRET_LABEL_USED = secret_label
                break
    except Exception:
        GEMINI_API_KEY = ''
if GEMINI_API_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
LLM_ENABLED = bool(GEMINI_API_KEY)
LLM_MODEL = 'gemini-3.8-flash'
print('Gemini tool calling enabled:', LLM_ENABLED)
print('Secret source:', SECRET_LABEL_USED or 'not available')


In [ ]:
# EIG training/validation estimates use independent random seeds to reduce selection optimism.
from cosmic_black_swan.planner import expected_information_gain

REQUIRED_PARTICLE_FIELDS = ('phase', 'depth_primary', 'depth_ratio', 'duration_fraction')
with np.load(POSTERIOR_PATH) as archive:
    missing = sorted(set(REQUIRED_PARTICLE_FIELDS) - set(archive.files))
    if missing: raise ValueError(f'Day 9.19 posterior lacks planner fields: {missing}; found {archive.files}')
    particle_frame = pd.DataFrame({key: np.asarray(archive[key]).reshape(-1) for key in archive.files if key in {
        'period_days', 'phase', 'depth_primary', 'depth_ratio', 'duration_fraction',
        'ellipsoidal_amplitude', 'reflection_amplitude'
    }})
particle_frame = particle_frame.replace([np.inf, -np.inf], np.nan).dropna(subset=list(REQUIRED_PARTICLE_FIELDS))
if len(particle_frame) < 50: raise ValueError(f'Only {len(particle_frame)} usable posterior particles; need at least 50.')
# Fixed period is not stored in the conditional posterior because Day 9.19 held it fixed.
# Read it from the saved Day 9.19 case diagnostics to reconstruct the particle parameter table.
case_hits = list(INPUT.rglob('day9_19_case_diagnostics.csv'))
if not case_hits: raise FileNotFoundError('Day 9.19 case diagnostics CSV is missing.')
case_file = max(case_hits, key=lambda p: p.stat().st_size)
case_table = pd.read_csv(case_file)
row_case = case_table[case_table['simulation_row'].astype(int).eq(4923)]
if len(row_case) != 1: raise ValueError('Could not identify exactly one Day 9.19 row 4923 case record.')
period_fixed = float(row_case.iloc[0]['period_fixed_days'])
particle_frame['period_days'] = period_fixed
for name, default in (('ellipsoidal_amplitude', 0.0), ('reflection_amplitude', 0.0)):
    if name not in particle_frame: particle_frame[name] = default
rng = np.random.default_rng(20261104)
if len(particle_frame) > 250:
    particle_frame = particle_frame.iloc[rng.choice(len(particle_frame), 250, replace=False)].reset_index(drop=True)

# Small hypothetical action menu: same start/cadence, different duration and noise.
actions = pd.DataFrame([
    {'action_id': f'{days}d_noise{noise:g}', 'start_time_days': 27.0, 'duration_days': days,
     'cadence_days': 0.0104, 'photometric_noise': noise}
    for days in (1.0, 2.0, 3.0, 5.0, 7.0) for noise in (0.001, 0.002)
])
train_eig = expected_information_gain(particle_frame, actions, predictive_draws=32, seed=111)
validation_eig = expected_information_gain(particle_frame, actions, predictive_draws=128, seed=222)
selected_id = str(train_eig.iloc[0]['action_id'])
validation_sorted = validation_eig.sort_values('action_id').reset_index(drop=True)
selected = validation_sorted[validation_sorted.action_id.astype(str).eq(selected_id)].iloc[0]
random_expected = float(validation_sorted['expected_information_gain_nats'].mean())
selected_validation = float(selected['expected_information_gain_nats'])

OUT = WORK / 'day11-candidate-investigation'
OUT.mkdir(parents=True, exist_ok=True)
train_eig.to_csv(OUT / 'day11_eig_training_scores.csv', index=False)
validation_sorted.to_csv(OUT / 'day11_eig_validation_scores.csv', index=False)
summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'posterior_source': str(POSTERIOR_PATH),
    'case': 'Day 9.19 row 4923; period fixed at saved conditional mode',
    'posterior_particles_used': int(len(particle_frame)),
    'hypothetical_actions': int(len(actions)),
    'training_predictive_draws_per_action': 32,
    'independent_validation_draws_per_action': 128,
    'selected_action_from_training_estimate': selected_id,
    'selected_action_validation_eig_nats': selected_validation,
    'uniform_random_action_expected_validation_eig_nats': random_expected,
    'selected_minus_random_eig_nats': selected_validation - random_expected,
    'selected_action_near_particle_ceiling': bool(selected['near_particle_ceiling']),
    'scope': 'Synthetic posterior-predictive policy comparison over a small hypothetical action menu; no real follow-up outcomes were used.'
}
(OUT / 'day11_eig_comparison.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('EIG comparison:', json.dumps(summary, indent=2))
print('\nIndependent validation ranking:')
print(validation_sorted[['action_id', 'expected_information_gain_nats', 'monte_carlo_standard_error_nats', 'near_particle_ceiling']].to_string(index=False))


In [ ]:
import base64 as _base64
catalogs_path = PROJECT / 'src' / 'cosmic_black_swan' / 'catalogs.py'
dossier_path = PROJECT / 'src' / 'cosmic_black_swan' / 'dossier.py'
catalogs_source = _base64.b64decode("IiIiVHJhY2VhYmxlIGxpdmUgU0lNQkFEIGFuZCBHYWlhIERSMyB2aWEgVml6aWVSIHF1ZXJpZXMuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjc3YKaW1wb3J0IGlvCmltcG9ydCB0aW1lCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQoKaW1wb3J0IG51bXB5IGFzIG5wCgoKZGVmIF9wbGFpbih2YWx1ZSk6CiAgICBpZiB2YWx1ZSBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAobnAuaW50ZWdlciwpKToKICAgICAgICByZXR1cm4gaW50KHZhbHVlKQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgKG5wLmZsb2F0aW5nLCkpOgogICAgICAgIHJldHVybiBmbG9hdCh2YWx1ZSkgaWYgbnAuaXNmaW5pdGUodmFsdWUpIGVsc2UgTm9uZQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgYnl0ZXMpOgogICAgICAgIHJldHVybiB2YWx1ZS5kZWNvZGUoInV0Zi04IiwgZXJyb3JzPSJyZXBsYWNlIikKICAgIGlmIGhhc2F0dHIodmFsdWUsICJpc29mb3JtYXQiKToKICAgICAgICByZXR1cm4gdmFsdWUuaXNvZm9ybWF0KCkKICAgIHRyeToKICAgICAgICBpZiBib29sKG5wLm1hLmlzX21hc2tlZCh2YWx1ZSkpOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgZXhjZXB0IChUeXBlRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgIHBhc3MKICAgIHRyeToKICAgICAgICByZXR1cm4gdmFsdWUuaXRlbSgpCiAgICBleGNlcHQgKEF0dHJpYnV0ZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICByZXR1cm4gc3RyKHZhbHVlKQoKCmRlZiBfcm93cyh0YWJsZSwgbGltaXQ6IGludCA9IDIwKSAtPiBsaXN0W2RpY3RdOgogICAgaWYgdGFibGUgaXMgTm9uZToKICAgICAgICByZXR1cm4gW10KICAgIHJldHVybiBbe3N0cihuYW1lKTogX3BsYWluKHJvd1tuYW1lXSkgZm9yIG5hbWUgaW4gdGFibGUuY29sbmFtZXN9IGZvciByb3cgaW4gdGFibGVbOmxpbWl0XV0KCgpkZWYgX3BhcnNlX2FzdV90c3YodGV4dDogc3RyKSAtPiBsaXN0W2RpY3RdOgogICAgbGluZXMgPSBbbGluZSBmb3IgbGluZSBpbiB0ZXh0LnNwbGl0bGluZXMoKSBpZiBsaW5lLnN0cmlwKCkgYW5kIG5vdCBsaW5lLmxzdHJpcCgpLnN0YXJ0c3dpdGgoIiMiKV0KICAgIHBhcnNlZCA9IFtdCiAgICBmb3IgbGluZSBpbiBsaW5lczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGNlbGxzID0gbmV4dChjc3YucmVhZGVyKFtsaW5lXSwgZGVsaW1pdGVyPSJcdCIpKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIGNlbGxzID0gbGluZS5zcGxpdCgpCiAgICAgICAgcGFyc2VkLmFwcGVuZChbY2VsbC5zdHJpcCgpIGZvciBjZWxsIGluIGNlbGxzXSkKICAgIGhlYWRlcl9pbmRleCA9IG5leHQoKGkgZm9yIGksIGNlbGxzIGluIGVudW1lcmF0ZShwYXJzZWQpIGlmIGFueShjZWxsLmxvd2VyKCkgPT0gInNvdXJjZSIgZm9yIGNlbGwgaW4gY2VsbHMpKSwgTm9uZSkKICAgIGlmIGhlYWRlcl9pbmRleCBpcyBOb25lOgogICAgICAgIHJldHVybiBbXQogICAgaGVhZGVyID0gcGFyc2VkW2hlYWRlcl9pbmRleF0KICAgIHJlY29yZHMgPSBbXQogICAgZm9yIGNlbGxzIGluIHBhcnNlZFtoZWFkZXJfaW5kZXggKyAxOl06CiAgICAgICAgaWYgbm90IGNlbGxzIG9yIGFsbChub3QgY2VsbCBvciBzZXQoY2VsbCkgPD0geyItIiwgIiAifSBmb3IgY2VsbCBpbiBjZWxscyk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgbGVuKGNlbGxzKSAhPSBsZW4oaGVhZGVyKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICByZWNvcmRzLmFwcGVuZCh7a2V5OiB2YWx1ZSBmb3Iga2V5LCB2YWx1ZSBpbiB6aXAoaGVhZGVyLCBjZWxscyl9KQogICAgcmV0dXJuIHJlY29yZHNbOjIwXQoKCmRlZiBxdWVyeV9zaW1iYWQoc291cmNlX2lkOiBzdHIsIHJhX2RlZzogZmxvYXQgfCBOb25lID0gTm9uZSwgZGVjX2RlZzogZmxvYXQgfCBOb25lID0gTm9uZSkgLT4gZGljdDoKICAgICIiIk1ha2UgYSBsaXZlIDUgYXJjc2VjIFNJTUJBRCBjb29yZGluYXRlIHF1ZXJ5IGF0IHRoZSBIYXJ2YXJkIG1pcnJvci4iIiIKICAgIGV2aWRlbmNlX2lkID0gZiJTSU1CQUQ6e3NvdXJjZV9pZH0iCiAgICBiYXNlID0gewogICAgICAgICJldmlkZW5jZV9pZCI6IGV2aWRlbmNlX2lkLAogICAgICAgICJjYXRhbG9nIjogIlNJTUJBRCIsCiAgICAgICAgInF1ZXJpZWRfYXRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksCiAgICAgICAgInF1ZXJ5IjogZiJUSUMge3NvdXJjZV9pZH0iLAogICAgICAgICJxdWVyeV9tZXRob2QiOiAibGl2ZSA1IGFyY3NlYyBjb29yZGluYXRlIGNvbmUiLAogICAgICAgICJtaXJyb3IiOiAic2ltYmFkLmhhcnZhcmQuZWR1IiwKICAgIH0KICAgIGlmIHJhX2RlZyBpcyBOb25lIG9yIGRlY19kZWcgaXMgTm9uZSBvciBub3QgbnAuaXNmaW5pdGUoW3JhX2RlZywgZGVjX2RlZ10pLmFsbCgpOgogICAgICAgIHJldHVybiB7KipiYXNlLCAic3RhdHVzIjogInF1ZXJ5X2ZhaWxlZCIsICJlcnJvciI6ICJObyBmaW5pdGUgY2FuZGlkYXRlIGNvb3JkaW5hdGVzIHdlcmUgYXZhaWxhYmxlLiJ9CiAgICB0cnk6CiAgICAgICAgZnJvbSBhc3Ryb3B5IGltcG9ydCB1bml0cyBhcyB1CiAgICAgICAgZnJvbSBhc3Ryb3B5LmNvb3JkaW5hdGVzIGltcG9ydCBTa3lDb29yZAogICAgICAgIGZyb20gYXN0cm9xdWVyeS5zaW1iYWQgaW1wb3J0IFNpbWJhZAogICAgICAgIHNpbWJhZCA9IFNpbWJhZCgpCiAgICAgICAgc2ltYmFkLnNlcnZlciA9ICJzaW1iYWQuaGFydmFyZC5lZHUiCiAgICAgICAgc2ltYmFkLnRpbWVvdXQgPSAxMgogICAgICAgIGNvb3JkID0gU2t5Q29vcmQoZmxvYXQocmFfZGVnKSAqIHUuZGVnLCBmbG9hdChkZWNfZGVnKSAqIHUuZGVnLCBmcmFtZT0iaWNycyIpCiAgICAgICAgcmVzdWx0ID0gc2ltYmFkLnF1ZXJ5X3JlZ2lvbihjb29yZCwgcmFkaXVzPTUuMCAqIHUuYXJjc2VjKQogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICoqYmFzZSwKICAgICAgICAgICAgInN0YXR1cyI6ICJxdWVyeV9zdWNjZWVkZWQiLAogICAgICAgICAgICAiY2VudGVyX2ljcnNfZGVnIjogeyJyYSI6IGZsb2F0KHJhX2RlZyksICJkZWMiOiBmbG9hdChkZWNfZGVnKX0sCiAgICAgICAgICAgICJyZWNvcmRfY291bnRfcmV0dXJuZWQiOiAwIGlmIHJlc3VsdCBpcyBOb25lIGVsc2UgaW50KGxlbihyZXN1bHQpKSwKICAgICAgICAgICAgInJlY29yZHMiOiBfcm93cyhyZXN1bHQpLAogICAgICAgICAgICAic291cmNlX3VybCI6ICJodHRwczovL3NpbWJhZC5oYXJ2YXJkLmVkdS9zaW1iYWQvc2ltLXNjcmlwdCIsCiAgICAgICAgfQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmV0dXJuIHsqKmJhc2UsICJzdGF0dXMiOiAicXVlcnlfZmFpbGVkIiwgImVycm9yIjogZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge3N0cihleGMpWzo0MDBdfSJ9CgoKZGVmIHF1ZXJ5X2dhaWFfdml6aWVyKHJhX2RlZzogZmxvYXQsIGRlY19kZWc6IGZsb2F0LCByYWRpdXNfYXJjc2VjOiBmbG9hdCA9IDUuMCkgLT4gZGljdDoKICAgICIiIlF1ZXJ5IFZpemllUidzIGxpdmUgR2FpYSBEUjMgQVNVIFRTViBlbmRwb2ludDsgcmV0cnkgb25jZSBhZnRlciBhIHRpbWVvdXQuIiIiCiAgICBpbXBvcnQgcmVxdWVzdHMKICAgIGlmIG5vdCAtOTAuMCA8PSBmbG9hdChkZWNfZGVnKSA8PSA5MC4wIG9yIG5vdCAwLjAgPD0gZmxvYXQocmFfZGVnKSA8IDM2MC4wOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNvb3JkaW5hdGVzIG11c3QgYmUgSUNSUyBkZWdyZWVzIikKICAgIGlmIG5vdCAwLjEgPD0gZmxvYXQocmFkaXVzX2FyY3NlYykgPD0gMzAuMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJGb3IgdGhpcyBjYW5kaWRhdGUgYXVkaXQsIHJhZGl1c19hcmNzZWMgbXVzdCBiZSB3aXRoaW4gMC4xIHRvIDMwIGFyY3NlYyIpCiAgICBldmlkZW5jZV9pZCA9IGYiVklaSUVSLUdBSUEtRFIzOntyYV9kZWc6LjZmfTp7ZGVjX2RlZzouNmZ9IgogICAgcXVlcmllZCA9IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpCiAgICB1cmwgPSAiaHR0cHM6Ly92aXppZXIuY2ZhLmhhcnZhcmQuZWR1L3Zpei1iaW4vYXN1LXRzdiIKICAgIHBhcmFtcyA9IHsKICAgICAgICAiLXNvdXJjZSI6ICJJLzM1NS9nYWlhZHIzIiwKICAgICAgICAiLWMucmEiOiBmIntmbG9hdChyYV9kZWcpOi44Zn0iLAogICAgICAgICItYy5kZWMiOiBmIntmbG9hdChkZWNfZGVnKTouOGZ9IiwKICAgICAgICAiLWMucnMiOiBmIntmbG9hdChyYWRpdXNfYXJjc2VjKTpnfSIsCiAgICAgICAgIi1vdXQiOiAiU291cmNlLFJBX0lDUlMsREVfSUNSUyxHbWFnIiwKICAgICAgICAiLW91dC5tYXgiOiAiNSIsCiAgICB9CiAgICBlcnJvcnMgPSBbXQogICAgZm9yIHRpbWVvdXQgaW4gKCg1LCAxMiksICg1LCAyMCkpOgogICAgICAgIHN0YXJ0ZWQgPSB0aW1lLnRpbWUoKQogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzcG9uc2UgPSByZXF1ZXN0cy5nZXQoCiAgICAgICAgICAgICAgICB1cmwsCiAgICAgICAgICAgICAgICBwYXJhbXM9cGFyYW1zLAogICAgICAgICAgICAgICAgaGVhZGVycz17IlVzZXItQWdlbnQiOiAiQ29zbWljU3dhbkxpdmVDYXRhbG9nQ2hlY2svMS4wIn0sCiAgICAgICAgICAgICAgICB0aW1lb3V0PXRpbWVvdXQsCiAgICAgICAgICAgICkKICAgICAgICAgICAgcmVzcG9uc2UucmFpc2VfZm9yX3N0YXR1cygpCiAgICAgICAgICAgIHRleHQgPSByZXNwb25zZS50ZXh0CiAgICAgICAgICAgIGlmICJTb3VyY2UiIG5vdCBpbiB0ZXh0IG9yICJnYWlhZHIzIiBub3QgaW4gdGV4dC5sb3dlcigpIG9yICI8aHRtbCIgaW4gdGV4dFs6NTAwXS5sb3dlcigpOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVW5leHBlY3RlZCBWaXppZVIgVFNWIHJlc3BvbnNlIChIVFRQIHtyZXNwb25zZS5zdGF0dXNfY29kZX0sIHtsZW4odGV4dCl9IGNoYXJhY3RlcnMpIikKICAgICAgICAgICAgcmVjb3JkcyA9IF9wYXJzZV9hc3VfdHN2KHRleHQpCiAgICAgICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICAgICAiZXZpZGVuY2VfaWQiOiBldmlkZW5jZV9pZCwKICAgICAgICAgICAgICAgICJjYXRhbG9nIjogIlZpemllUiBJLzM1NS9nYWlhZHIzIChHYWlhIERSMykiLAogICAgICAgICAgICAgICAgInN0YXR1cyI6ICJxdWVyeV9zdWNjZWVkZWQiLAogICAgICAgICAgICAgICAgInF1ZXJpZWRfYXRfdXRjIjogcXVlcmllZCwKICAgICAgICAgICAgICAgICJxdWVyeV9tZXRob2QiOiAibGl2ZSBWaXppZVIgQVNVIFRTViBjb25lIHNlYXJjaCIsCiAgICAgICAgICAgICAgICAibWlycm9yIjogInZpemllci5jZmEuaGFydmFyZC5lZHUiLAogICAgICAgICAgICAgICAgImVuZHBvaW50IjogIi92aXotYmluL2FzdS10c3YiLAogICAgICAgICAgICAgICAgImNlbnRlcl9pY3JzX2RlZyI6IHsicmEiOiBmbG9hdChyYV9kZWcpLCAiZGVjIjogZmxvYXQoZGVjX2RlZyl9LAogICAgICAgICAgICAgICAgInJhZGl1c19hcmNzZWMiOiBmbG9hdChyYWRpdXNfYXJjc2VjKSwKICAgICAgICAgICAgICAgICJodHRwX3N0YXR1cyI6IGludChyZXNwb25zZS5zdGF0dXNfY29kZSksCiAgICAgICAgICAgICAgICAicmVjb3JkX2NvdW50X3JldHVybmVkIjogbGVuKHJlY29yZHMpLAogICAgICAgICAgICAgICAgInJlY29yZHMiOiByZWNvcmRzLAogICAgICAgICAgICAgICAgInJhd19yZXNwb25zZV9wcmV2aWV3IjogdGV4dFs6MTgwMF0sCiAgICAgICAgICAgICAgICAic291cmNlX3VybCI6IHJlc3BvbnNlLnVybCwKICAgICAgICAgICAgICAgICJhdHRlbXB0X2Vycm9yc19iZWZvcmVfc3VjY2VzcyI6IGVycm9ycywKICAgICAgICAgICAgICAgICJzZWNvbmRzIjogcm91bmQodGltZS50aW1lKCkgLSBzdGFydGVkLCAyKSwKICAgICAgICAgICAgfQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKHsiZXJyb3IiOiBmInt0eXBlKGV4YykuX19uYW1lX199OiB7c3RyKGV4YylbOjM1MF19IiwgInRpbWVvdXRfc2Vjb25kcyI6IHRpbWVvdXRbMV19KQogICAgcmV0dXJuIHsKICAgICAgICAiZXZpZGVuY2VfaWQiOiBldmlkZW5jZV9pZCwKICAgICAgICAiY2F0YWxvZyI6ICJWaXppZVIgSS8zNTUvZ2FpYWRyMyAoR2FpYSBEUjMpIiwKICAgICAgICAic3RhdHVzIjogInF1ZXJ5X2ZhaWxlZCIsCiAgICAgICAgInF1ZXJpZWRfYXRfdXRjIjogcXVlcmllZCwKICAgICAgICAicXVlcnlfbWV0aG9kIjogImxpdmUgVml6aWVSIEFTVSBUU1YgY29uZSBzZWFyY2giLAogICAgICAgICJtaXJyb3IiOiAidml6aWVyLmNmYS5oYXJ2YXJkLmVkdSIsCiAgICAgICAgImNlbnRlcl9pY3JzX2RlZyI6IHsicmEiOiBmbG9hdChyYV9kZWcpLCAiZGVjIjogZmxvYXQoZGVjX2RlZyl9LAogICAgICAgICJyYWRpdXNfYXJjc2VjIjogZmxvYXQocmFkaXVzX2FyY3NlYyksCiAgICAgICAgInJlY29yZF9jb3VudF9yZXR1cm5lZCI6IE5vbmUsCiAgICAgICAgInJlY29yZHMiOiBbXSwKICAgICAgICAiYXR0ZW1wdF9lcnJvcnMiOiBlcnJvcnMsCiAgICAgICAgImludGVycHJldGF0aW9uIjogIlRoZSBsaXZlIHNlcnZpY2UgcmVxdWVzdCB0aW1lZCBvdXQgb3IgZXJyb3JlZDsgdGhpcyBpcyBub3QgYSB6ZXJvLW1hdGNoIHJlc3VsdC4iLAogICAgICAgICJzb3VyY2VfdXJsIjogdXJsLAogICAgfQoKCmRlZiBxdWVyeV9jYXRhbG9ncyhzb3VyY2VfaWQ6IHN0ciwgcmFfZGVnOiBmbG9hdCB8IE5vbmUsIGRlY19kZWc6IGZsb2F0IHwgTm9uZSwgcmFkaXVzX2FyY3NlYzogZmxvYXQgPSA1LjApIC0+IGxpc3RbZGljdF06CiAgICBldmlkZW5jZSA9IFtxdWVyeV9zaW1iYWQoc291cmNlX2lkLCByYV9kZWcsIGRlY19kZWcpXQogICAgaWYgcmFfZGVnIGlzIG5vdCBOb25lIGFuZCBkZWNfZGVnIGlzIG5vdCBOb25lIGFuZCBucC5pc2Zpbml0ZShbcmFfZGVnLCBkZWNfZGVnXSkuYWxsKCk6CiAgICAgICAgZXZpZGVuY2UuYXBwZW5kKHF1ZXJ5X2dhaWFfdml6aWVyKGZsb2F0KHJhX2RlZyksIGZsb2F0KGRlY19kZWcpLCByYWRpdXNfYXJjc2VjKSkKICAgIHJldHVybiBldmlkZW5jZQo=").decode('utf-8')
dossier_source = _base64.b64decode("...").decode('utf-8')
catalogs_path.write_text(catalogs_source, encoding='utf-8')
dossier_path.write_text(dossier_source, encoding='utf-8')
print('[PASS] Manual Gemini tools loaded with low thinking and ID-preserving function responses.')


In [ ]:
# One-candidate retry: preserve live catalog evidence even if Gemini times out again.
from cosmic_black_swan.dossier import build_dossier

if LLM_ENABLED:
    if importlib.util.find_spec('astroquery') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'astroquery'], check=True, timeout=240)
    if importlib.util.find_spec('pyvo') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvo'], check=True, timeout=180)
    try:
        import google.genai  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'google-genai>=1.0'], check=True, timeout=180)

REVIEWED_TICS = ['11521787', '30317282', '3996132', '4637806']
features['source_id'] = features['source_id'].astype(str).str.replace(r'\.0$', '', regex=True)
available_ids = set(features['source_id'].dropna())
selected_ids = [tic for tic in REVIEWED_TICS if tic in available_ids][:1]
if not selected_ids:
    raise RuntimeError('No reviewed TIC is present in the selected checkpoint.')
print('Retry target:', selected_ids, '| Gemini tool calling:', LLM_ENABLED)

dossier_rows = []
if not LLM_ENABLED:
    raise RuntimeError('Gemini-Api-Key is not attached to this notebook. Enable it in Add-ons > Secrets.')
for tic in selected_ids:
    started = time.time()
    row = {'source_id': tic, 'status': 'dossier_failed'}
    try:
        result = build_dossier(DATA, tic, with_llm=True, llm_model=LLM_MODEL, radius_arcsec=5.0)
        dossier_path = Path(result['json'])
        dossier = json.loads(dossier_path.read_text(encoding='utf-8'))
        llm = dossier.get('llm') or {}
        narrative = llm.get('narrative') or {}
        claims = narrative.get('claims', []) if isinstance(narrative, dict) else []
        traces = dossier.get('catalog_queries', [])
        valid_claims = sum(bool(claim.get('citation_ids_valid')) for claim in claims if isinstance(claim, dict))
        simbad_status = next((item.get('status') for item in traces if item.get('catalog') == 'SIMBAD'), 'not_called')
        vizier_status = next((item.get('status') for item in traces if 'VizieR' in str(item.get('catalog', ''))), 'not_called')
        live_successes = sum(item.get('status') == 'query_succeeded' for item in traces)
        llm_ok = llm.get('status') == 'generated'
        row.update({
            'status': 'dossier_created' if llm_ok else 'catalog_dossier_created_gemini_failed',
            'dossier_json': str(dossier_path), 'dossier_markdown': result.get('markdown'),
            'gemini_status': llm.get('status', 'unknown'), 'gemini_error': llm.get('error'),
            'gemini_tool_calls': llm.get('gemini_tool_calls', 0), 'catalog_evidence_mode': llm.get('evidence_mode'),
            'simbad_live_status': simbad_status, 'vizier_live_status': vizier_status,
            'successful_live_catalog_queries': int(live_successes), 'llm_claims': len(claims),
            'claims_with_valid_citation_ids': valid_claims,
            'citation_id_validity_fraction': valid_claims / len(claims) if claims else None,
            'pixel_audit_status': (dossier.get('pixel_audit') or {}).get('status'),
        })
    except Exception as exc:
        row.update({'status': 'dossier_failed', 'error': f'{type(exc).__name__}: {str(exc)[:500]}'})
        print('Dossier failed for', tic, row['error'])
    row['seconds'] = round(time.time() - started, 1)
    dossier_rows.append(row)
    print(json.dumps(row, indent=2, default=str), flush=True)

OUT.mkdir(parents=True, exist_ok=True)
dossier_dir = DATA / 'dossiers'
retry_dir = OUT / 'live_dossiers'
retry_dir.mkdir(parents=True, exist_ok=True)
for tic in selected_ids:
    for suffix in ('.json', '.md'):
        candidate_file = dossier_dir / f'TIC_{tic}{suffix}'
        if candidate_file.is_file():
            shutil.copy2(candidate_file, retry_dir / candidate_file.name)
if any(retry_dir.iterdir()):
    bundle = shutil.make_archive(str(OUT / 'day11_live_candidate_dossiers'), 'zip', root_dir=retry_dir)
    print('[PASS] Dossier bundle:', bundle)
else:
    print('[FAIL] No dossier files were saved.')

dossier_audit = pd.DataFrame(dossier_rows)
dossier_audit.to_csv(OUT / 'day11_live_dossier_audit.csv', index=False)
annotation_template = pd.DataFrame([{'source_id': tic, 'required_evidence_ids': '', 'reviewed_claims': '', 'supported_claims': ''} for tic in selected_ids])
annotation_template.to_csv(OUT / 'day11_dossier_human_review_template.csv', index=False)
report_lines = [
    '# Day 11 live candidate investigation retry', '',
    f"- EIG selected action: `{summary['selected_action_from_training_estimate']}`; validation EIG {summary['selected_action_validation_eig_nats']:.4f} nats.",
    '- The EIG comparison is synthetic and uses no real follow-up outcomes.',
    '- Live SIMBAD/VizieR query successes and failures are recorded per candidate; a timeout is not a non-detection.',
    '- If Gemini timed out, the saved dossier retains live catalog evidence and marks the narrative unavailable.',
    '- Human review is needed to judge whether generated claims are supported.', '',
    '| TIC | Status | Gemini | SIMBAD | VizieR | Gemini tool calls | Successful live queries | Seconds |',
    '|---|---|---|---|---|---:|---:|---:|'
]
for r in dossier_rows:
    report_lines.append(f"| {r['source_id']} | {r.get('status','')} | {r.get('gemini_status','')} | {r.get('simbad_live_status','')} | {r.get('vizier_live_status','')} | {r.get('gemini_tool_calls','')} | {r.get('successful_live_catalog_queries','')} | {r.get('seconds','')} |")
(OUT / 'day11_live_candidate_investigation_report.md').write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print('Saved retry outputs:', OUT)
print(dossier_audit.to_string(index=False))
